# 🧠 Mental Health Intelligence Platform

## Phase 6 — Data Preprocessing

### Overview

Machine learning models require structured numerical input.

This notebook transforms the engineered dataset into a machine-learning-ready format while preventing data leakage.

Unlike feature engineering, preprocessing is performed after splitting the dataset into training and testing sets to ensure that no information from the test set influences the training process.

---

### Objectives

- Prevent data leakage
- Handle structural missing values
- Encode categorical variables appropriately
- Scale numerical features
- Prepare datasets for machine learning

## Imports

In [60]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

import joblib

## Load Dataset

In [61]:
df = pd.read_csv("../data/processed/engineered_osmi_data.csv")

print(df.shape)

df.head()

(3082, 67)


,source_file,survey_year,self_employed,no_employees,tech_company,primary_role_tech,benefits,know_options,formal_discussion,resources,...,willing_to_interview,age,gender,country_live,state_live,race,country_work,state_work,age_group,workplace_support_score
0,2014,2014,NaN,6-25,Yes,NaN,Yes,Don't Know,No,Yes,...,NaN,NaN,Male,NaN,NaN,NaN,NaN,NaN,NaN,3
1,2014,2014,NaN,More than 1000,No,NaN,Don't Know,No,Don't Know,Don't Know,...,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2014,2014,NaN,6-25,Yes,NaN,No,No,No,No,...,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2014,2014,NaN,26-100,Yes,NaN,No,Yes,No,No,...,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2014,2014,NaN,100-500,Yes,NaN,Yes,No,Don't Know,Don't Know,...,NaN,NaN,Male,NaN,NaN,NaN,NaN,NaN,NaN,1


In [62]:
columns_to_drop = [
    "source_file",
    "additional_comments",
    "improvement_suggestions",
    "bring_up_mental_why"
]

# Drop the columns (errors='ignore' ensures it doesn't fail if a column is missing)
df = df.drop(columns=columns_to_drop, errors='ignore')

print(f"New shape after dropping columns: {df.shape}")
df.head()

New shape after dropping columns: (3082, 66)


,survey_year,self_employed,no_employees,tech_company,primary_role_tech,benefits,know_options,formal_discussion,resources,anonymity,...,willing_to_interview,age,gender,country_live,state_live,race,country_work,state_work,age_group,workplace_support_score
0,2014,NaN,6-25,Yes,NaN,Yes,Don't Know,No,Yes,Yes,...,NaN,NaN,Male,NaN,NaN,NaN,NaN,NaN,NaN,3
1,2014,NaN,More than 1000,No,NaN,Don't Know,No,Don't Know,Don't Know,Don't Know,...,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2014,NaN,6-25,Yes,NaN,No,No,No,No,Don't Know,...,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2014,NaN,26-100,Yes,NaN,No,Yes,No,No,No,...,NaN,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2014,NaN,100-500,Yes,NaN,Yes,No,Don't Know,Don't Know,Don't Know,...,NaN,NaN,Male,NaN,NaN,NaN,NaN,NaN,NaN,1


In [63]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3082 entries, 0 to 3081
Data columns (total 66 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   survey_year                        3082 non-null   int64  
 1   self_employed                      1228 non-null   str    
 2   no_employees                       2823 non-null   str    
 3   tech_company                       2823 non-null   str    
 4   primary_role_tech                  1577 non-null   str    
 5   benefits                           2180 non-null   str    
 6   know_options                       2660 non-null   str    
 7   formal_discussion                  2823 non-null   str    
 8   resources                          2823 non-null   str    
 9   anonymity                          2823 non-null   str    
 10  leave_ease                         2823 non-null   str    
 11  comfort_physical_mental            2823 non-null   str    
 12  com

## Feature Inventory

Before building the preprocessing pipeline, every feature is classified according to its role in the machine learning workflow.

The following categories will be identified:

- Target Variable
- Numeric Features
- Binary Features
- Ordinal Features
- Nominal Features
- Text Features
- Potential Leakage Features

This inventory ensures that each feature receives the appropriate preprocessing strategy.

In [64]:
numeric_features = df.select_dtypes(
    include=["int64", "float64", "Int64", "Float64"]
).columns.tolist()

print(f"Numeric Features ({len(numeric_features)})")

numeric_features

Numeric Features (12)


['survey_year',
 'employer_importance_physical',
 'employer_importance_mental',
 'have_prev_employers',
 'prev_employer_importance_physical',
 'prev_employer_importance_mental',
 'share_with_friends_family',
 'team_reaction_if_knew',
 'industry_support_rating',
 'willing_to_interview',
 'age',
 'workplace_support_score']

In [65]:
categorical_features = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print(f"Categorical Features ({len(categorical_features)})")

categorical_features

Categorical Features (54)


['self_employed',
 'no_employees',
 'tech_company',
 'primary_role_tech',
 'benefits',
 'know_options',
 'formal_discussion',
 'resources',
 'anonymity',
 'leave_ease',
 'comfort_physical_mental',
 'comfort_supervisor',
 'discussed_with_employer',
 'comfort_coworkers',
 'discussed_with_coworkers',
 'coworker_discussed_with_you',
 'coverage_mental_health',
 'know_resources',
 'reveal_to_clients',
 'reveal_to_coworkers',
 'productivity_affected',
 'productivity_percent_affected',
 'prev_tech_company',
 'prev_benefits',
 'prev_know_options',
 'prev_formal_discussion',
 'prev_resources',
 'prev_anonymity',
 'prev_comfort_physical_mental',
 'prev_comfort_supervisor',
 'prev_discussed_with_employer',
 'prev_comfort_coworkers',
 'prev_discussed_with_coworkers',
 'currently_disordered',
 'diagnosed_disorder',
 'past_disorder',
 'sought_treatment',
 'family_history',
 'interference_when_treated',
 'interference_when_not_treated',
 'observations_less_likely_reveal',
 'bring_up_physical_interview

In [66]:
print("="*80)
print("BINARY CANDIDATES")
print("="*80)

for col in categorical_features:

    values = sorted(df[col].dropna().astype(str).unique())

    if len(values) <= 4:

        print(f"\n{col}")

        print(values)

BINARY CANDIDATES

self_employed
['No', 'Yes']

tech_company
['No', 'Yes']

primary_role_tech
['No', 'Yes']

benefits
["Don't Know", 'No', 'Not Eligible', 'Yes']

know_options
["Don't Know", 'No', 'Yes']

formal_discussion
["Don't Know", 'No', 'Yes']

resources
["Don't Know", 'No', 'Yes']

anonymity
["Don't Know", 'No', 'Yes']

comfort_supervisor
['Maybe', 'No', 'Some of them', 'Yes']

discussed_with_employer
['Maybe', 'No', 'Yes']

comfort_coworkers
['Maybe', 'No', 'Some of them', 'Yes']

discussed_with_coworkers
['No', 'Yes']

coworker_discussed_with_you
['No', 'Yes']

coverage_mental_health
['No', 'Yes']

know_resources
['I know some', "No, I don't know any", 'Yes, I know several']

productivity_affected
['No', 'Not applicable to me', 'Unsure', 'Yes']

productivity_percent_affected
['1-25%', '26-50%', '51-75%', '76-100%']

prev_tech_company
['No', 'Yes']

prev_benefits
["Don't Know", 'No, none did', 'Some did', 'Yes, they all did']

prev_formal_discussion
["Don't Know", 'None did', 

In [67]:
ordinal_candidates = [

    "leave_ease",

    "comfort_supervisor",

    "comfort_coworkers",

    "comfort_physical_mental",

    "industry_support_rating"

]

for col in ordinal_candidates:

    if col in df.columns:

        print("="*80)

        print(col)

        print(df[col].value_counts(dropna=False))

leave_ease
leave_ease
Don't Know                    822
Somewhat easy                 688
Very easy                     546
Somewhat difficult            325
NaN                           259
Very difficult                243
Neither easy nor difficult    199
Name: count, dtype: int64
comfort_supervisor
comfort_supervisor
Yes             1090
No               868
Maybe            517
Some of them     348
NaN              259
Name: count, dtype: int64
comfort_coworkers
comfort_coworkers
Some of them    767
Yes             697
Maybe           683
No              676
NaN             259
Name: count, dtype: int64
comfort_physical_mental
comfort_physical_mental
Physical health                   1070
Don't know                         570
Same level of comfort for each     479
Yes                                339
No                                 337
NaN                                259
Mental health                       28
Name: count, dtype: int64
industry_support_rating
industry_sup

In [68]:
cardinality = pd.DataFrame({

    "Unique Values": df.nunique()

})

cardinality = cardinality.sort_values(

    "Unique Values",

    ascending=False

)

cardinality.head(20)

,Unique Values
country_work,58
age,50
state_work,44
state_live,15
team_reaction_if_knew,11
share_with_friends_family,11
employer_importance_physical,11
employer_importance_mental,11
prev_employer_importance_mental,11
prev_employer_importance_physical,11


In [69]:
missing = pd.DataFrame({

    "Missing": df.isna().sum(),

    "Percent": (df.isna().mean()*100).round(2)

})

missing = missing[missing["Missing"] > 0]

missing.sort_values(

    "Percent",

    ascending=False

)

,Missing,Percent
productivity_percent_affected,2888,93.71
identified_affected_career,2860,92.80
reveal_to_clients,2823,91.60
know_resources,2823,91.60
coverage_mental_health,2823,91.60
...,...,...
anonymity,259,8.40
leave_ease,259,8.40
comfort_physical_mental,259,8.40
comfort_supervisor,259,8.40


In [70]:
potential_leakage = [

    "currently_disordered",

    "diagnosed_disorder",

    "interference_when_treated",

    "interference_when_not_treated",

    "past_disorder"

]

for col in potential_leakage:

    if col in df.columns:

        print(col)

currently_disordered
diagnosed_disorder
interference_when_treated
interference_when_not_treated
past_disorder


## Feature Classification

Based on the dataset inspection, each feature is assigned a preprocessing strategy.

Features are divided into four categories:

- Numeric Features
- Binary Features
- Ordinal Features
- Nominal Features

Each category will receive a different preprocessing pipeline.

In [71]:
TARGET = "sought_treatment"

X = df.drop(columns=[TARGET])

y = df[TARGET]

In [72]:
leakage_features = [
    "currently_disordered",
    "diagnosed_disorder",
    "interference_when_treated",
    "interference_when_not_treated"
]

existing = [c for c in leakage_features if c in X.columns]

X = X.drop(columns=existing)

print("Dropped leakage features:")
print(existing)

Dropped leakage features:
['currently_disordered', 'diagnosed_disorder', 'interference_when_treated', 'interference_when_not_treated']


## Train/Test Split

In [73]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(2465, 61)
(617, 61)


In [74]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "Int64", "Float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

In [75]:
binary_features = []

for col in categorical_features:

    unique_values = sorted(
        X_train[col].dropna().astype(str).unique()
    )

    if set(unique_values).issubset({"Yes", "No"}):
        binary_features.append(col)

print(binary_features)

['self_employed', 'tech_company', 'primary_role_tech', 'discussed_with_coworkers', 'coworker_discussed_with_you', 'coverage_mental_health', 'prev_tech_company', 'prev_discussed_with_employer', 'prev_discussed_with_coworkers', 'openly_identified', 'identified_affected_career']


In [76]:
ordinal_features = [
    "leave_ease",
    "comfort_supervisor",
    "comfort_coworkers",
    "comfort_physical_mental"
]

ordinal_features = [
    c for c in ordinal_features
    if c in X_train.columns
]

In [77]:
nominal_features = [

    c for c in categorical_features

    if c not in binary_features
    and c not in ordinal_features

]

print(len(nominal_features))

34


In [78]:
binary_map = {
    "Yes": 1,
    "No": 0
}
leave_order = [
    "Very difficult",
    "Somewhat difficult",
    "Neither easy nor difficult",
    "Somewhat easy",
    "Very easy",
    "Don't know",
    "Not Asked"
]

In [79]:
print(df["leave_ease"].value_counts(dropna=False))

leave_ease
Don't Know                    822
Somewhat easy                 688
Very easy                     546
Somewhat difficult            325
NaN                           259
Very difficult                243
Neither easy nor difficult    199
Name: count, dtype: int64


In [80]:
print(df["comfort_supervisor"].value_counts(dropna=False))

comfort_supervisor
Yes             1090
No               868
Maybe            517
Some of them     348
NaN              259
Name: count, dtype: int64


In [81]:
print(df["comfort_coworkers"].value_counts(dropna=False))

comfort_coworkers
Some of them    767
Yes             697
Maybe           683
No              676
NaN             259
Name: count, dtype: int64


In [82]:
print(df["comfort_physical_mental"].value_counts(dropna=False))

comfort_physical_mental
Physical health                   1070
Don't know                         570
Same level of comfort for each     479
Yes                                339
No                                 337
NaN                                259
Mental health                       28
Name: count, dtype: int64


In [83]:
print("NUMERIC FEATURES")
print(numeric_features)

NUMERIC FEATURES
['survey_year', 'employer_importance_physical', 'employer_importance_mental', 'have_prev_employers', 'prev_employer_importance_physical', 'prev_employer_importance_mental', 'share_with_friends_family', 'team_reaction_if_knew', 'industry_support_rating', 'willing_to_interview', 'age', 'workplace_support_score']


In [84]:
print("BINARY FEATURES")
print(binary_features)

BINARY FEATURES
['self_employed', 'tech_company', 'primary_role_tech', 'discussed_with_coworkers', 'coworker_discussed_with_you', 'coverage_mental_health', 'prev_tech_company', 'prev_discussed_with_employer', 'prev_discussed_with_coworkers', 'openly_identified', 'identified_affected_career']


In [85]:
print("ORDINAL FEATURES")
print(ordinal_features)

ORDINAL FEATURES
['leave_ease', 'comfort_supervisor', 'comfort_coworkers', 'comfort_physical_mental']


In [86]:
print("NOMINAL FEATURES")
print(nominal_features)

NOMINAL FEATURES
['no_employees', 'benefits', 'know_options', 'formal_discussion', 'resources', 'anonymity', 'discussed_with_employer', 'know_resources', 'reveal_to_clients', 'reveal_to_coworkers', 'productivity_affected', 'productivity_percent_affected', 'prev_benefits', 'prev_know_options', 'prev_formal_discussion', 'prev_resources', 'prev_anonymity', 'prev_comfort_physical_mental', 'prev_comfort_supervisor', 'prev_comfort_coworkers', 'past_disorder', 'family_history', 'observations_less_likely_reveal', 'bring_up_physical_interview', 'bring_up_mental_interview', 'observed_unsupportive_response', 'observed_supportive_response', 'gender', 'country_live', 'state_live', 'race', 'country_work', 'state_work', 'age_group']


In [87]:
all_features = (
    numeric_features +
    binary_features +
    ordinal_features +
    nominal_features
)

duplicates = [x for x in set(all_features) if all_features.count(x) > 1]

print("Duplicate feature assignments:")
print(duplicates)

Duplicate feature assignments:
[]


In [88]:
for col in ordinal_features:

    print("="*70)
    print(col)
    print(X_train[col].dtype)
    print(X_train[col].unique())

leave_ease
str
<StringArray>
[             'Somewhat easy',                          nan,
                  'Very easy',                 'Don't Know',
         'Somewhat difficult',             'Very difficult',
 'Neither easy nor difficult']
Length: 7, dtype: str
comfort_supervisor
str
<StringArray>
['Yes', 'Maybe', nan, 'No', 'Some of them']
Length: 5, dtype: str
comfort_coworkers
str
<StringArray>
['Maybe', 'Yes', nan, 'No', 'Some of them']
Length: 5, dtype: str
comfort_physical_mental
str
<StringArray>
[               'Physical health',                              nan,
                             'No',                     'Don't know',
 'Same level of comfort for each',                            'Yes',
                  'Mental health']
Length: 7, dtype: str


In [89]:
# Combine processed features and target
final_df = X.copy()
final_df[TARGET] = y

print(final_df.shape)

final_df.head()

(3082, 62)


,survey_year,self_employed,no_employees,tech_company,primary_role_tech,benefits,know_options,formal_discussion,resources,anonymity,...,age,gender,country_live,state_live,race,country_work,state_work,age_group,workplace_support_score,sought_treatment
0,2014,NaN,6-25,Yes,NaN,Yes,Don't Know,No,Yes,Yes,...,NaN,Male,NaN,NaN,NaN,NaN,NaN,NaN,3,Yes
1,2014,NaN,More than 1000,No,NaN,Don't Know,No,Don't Know,Don't Know,Don't Know,...,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0,No
2,2014,NaN,6-25,Yes,NaN,No,No,No,No,Don't Know,...,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0,No
3,2014,NaN,26-100,Yes,NaN,No,Yes,No,No,No,...,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0,Yes
4,2014,NaN,100-500,Yes,NaN,Yes,No,Don't Know,Don't Know,Don't Know,...,NaN,Male,NaN,NaN,NaN,NaN,NaN,NaN,1,No


In [90]:
from pathlib import Path

Path("../data/processed").mkdir(
    parents=True,
    exist_ok=True
)

final_df.to_csv(
    "../data/processed/ml_ready_dataset.csv",
    index=False
)

print("✅ ML-ready dataset saved successfully!")

✅ ML-ready dataset saved successfully!


In [91]:
saved = pd.read_csv("../data/processed/ml_ready_dataset.csv")

print(saved.shape)

saved.head()

(3082, 62)


,survey_year,self_employed,no_employees,tech_company,primary_role_tech,benefits,know_options,formal_discussion,resources,anonymity,...,age,gender,country_live,state_live,race,country_work,state_work,age_group,workplace_support_score,sought_treatment
0,2014,NaN,6-25,Yes,NaN,Yes,Don't Know,No,Yes,Yes,...,NaN,Male,NaN,NaN,NaN,NaN,NaN,NaN,3,Yes
1,2014,NaN,More than 1000,No,NaN,Don't Know,No,Don't Know,Don't Know,Don't Know,...,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0,No
2,2014,NaN,6-25,Yes,NaN,No,No,No,No,Don't Know,...,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0,No
3,2014,NaN,26-100,Yes,NaN,No,Yes,No,No,No,...,NaN,Other,NaN,NaN,NaN,NaN,NaN,NaN,0,Yes
4,2014,NaN,100-500,Yes,NaN,Yes,No,Don't Know,Don't Know,Don't Know,...,NaN,Male,NaN,NaN,NaN,NaN,NaN,NaN,1,No
